# Caderno 02 — Análise Exploratória, O Paradoxo Disciplinar e Comparação Interdivisões

**Projeto:** Impacto das Apostas Esportivas no Futebol Brasileiro  
**Fase:** Fase 9 — Cadernos Executáveis e Reprodutibilidade  
**Data:** 2026-09-10  
**Autor:** Agente Antigravity (Advanced Agentic Coding)  

---

## 1. Visão Geral e Hipóteses de Pesquisa

Este caderno reproduz a análise exploratória de dados (EDA) e os testes de quebra estrutural do futebol brasileiro:
1. **Quebra Estrutural Pós-2018:** Comparação formal entre o período pré-bets (**2014–2018**) e o período de alta exposição (**2022–2024**);
2. **O "Paradoxo Disciplinar":** Demonstração matemática da redução simultânea de faltas ($-19,3\%$) e explosão da taxa de conversão de faltas em cartões ($+37,1\%$);
3. **Contraste Interdivisões (Série A vs. Série B):** Análise do descompasso de severidade arbitral (Série A $+11,7\%$ mais severa em cartões);
4. **Gradiente de Dose-Resposta:** Comparação empírica de partidas por grau de exposição a casas de apostas.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

PROJECT_ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Configurar estilo visual dos gráficos
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.sans-serif"] = "DejaVu Sans"


## 2. Evolução Histórica das Métricas Disciplinares (Série A 2015–2024)

Analisamos o comportamento anual das faltas médias por jogo, cartões amarelos, vermelhos e da taxa de conversão $\tau_{\text{CF}}$.


In [ ]:
tabela_01_path = os.path.join(PROJECT_ROOT, "reports", "tables", "tabela_01_metricas_por_temporada.csv")
df_metricas = pd.read_csv(tabela_01_path)
df_metricas_sub = df_metricas[df_metricas["temporada"].between(2015, 2024)]
print("Métricas Disciplinares por Temporada (Série A 2015–2024):")
display(df_metricas_sub[["temporada", "cartoes_media", "faltas_media", "taxa_cartao_por_falta", "penaltis_por_jogo"]])


## 3. O Paradoxo Disciplinar: Queda de Faltas vs. Alta de Cartões

A teoria esportiva tradicional prevê que o número de cartões deve correlacionar-se positivamente com o volume de faltas cometidas. No entanto, o futebol brasileiro apresenta um **desacoplamento estrutural**:
* Em 2017: **31,41 faltas/jogo** e **4,97 cartões/jogo** (Taxa $\tau = 0{,}1582$);
* Em 2024: **25,36 faltas/jogo** e **5,52 cartões/jogo** (Taxa $\tau = 0{,}2198$).


In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))

color = "#1f77b4"
ax1.set_xlabel("Temporada", fontsize=12)
ax1.set_ylabel("Faltas Médias por Partida", color=color, fontsize=12)
line1 = ax1.plot(df_metricas_sub["temporada"], df_metricas_sub["faltas_media"], color=color, marker="o", linewidth=2.5, label="Faltas/Jogo")
ax1.tick_params(axis="y", labelcolor=color)

ax2 = ax1.twinx()
color = "#d62728"
ax2.set_ylabel("Taxa de Conversão (Cartões / Faltas)", color=color, fontsize=12)
line2 = ax2.plot(df_metricas_sub["temporada"], df_metricas_sub["taxa_cartao_por_falta"], color=color, marker="s", linewidth=2.5, linestyle="--", label="Taxa Conversão")
ax2.tick_params(axis="y", labelcolor=color)

plt.title("O Paradoxo Disciplinar no Futebol Brasileiro (Série A 2015–2024)", fontsize=14, fontweight="bold", pad=15)
fig.tight_layout()
plt.show()


## 4. Testes de Hipótese: Pré-Bets (2014–2018) vs. Pós-Bets (2022–2024)

Avaliamos formalmente se o aumento de cartões e da taxa de conversão entre o período basal e o período contemporâneo possui significância estatística.


In [ ]:
cartoes_path = os.path.join(PROJECT_ROOT, "data", "processed", "serie_a", "cartoes.parquet")
partidas_path = os.path.join(PROJECT_ROOT, "data", "processed", "serie_a", "partidas.parquet")
df_c = pd.read_parquet(cartoes_path)
df_p = pd.read_parquet(partidas_path)

cartoes_por_jogo = df_c.groupby("partida_id").size()
df_p["total_cartoes"] = df_p["partida_id"].map(cartoes_por_jogo).fillna(0)

pre_bets = df_p[df_p["temporada"].between(2014, 2018)]["total_cartoes"]
pos_bets = df_p[df_p["temporada"].between(2022, 2024)]["total_cartoes"]

t_stat, p_val_t = stats.ttest_ind(pre_bets, pos_bets, equal_var=False)
u_stat, p_val_u = stats.mannwhitneyu(pre_bets, pos_bets)

print(f"Média de Cartões Pré-Bets (2014-2018): {pre_bets.mean():.3f} (N={len(pre_bets)})")
print(f"Média de Cartões Pós-Bets (2022-2024): {pos_bets.mean():.3f} (N={len(pos_bets)})")
print(f"Diferença: +{(pos_bets.mean() - pre_bets.mean()):.3f} cartões/jogo (+{((pos_bets.mean() / pre_bets.mean()) - 1)*100:.2f}%)")
print(f"Teste t de Welch:  t = {t_stat:.4f}, p-valor = {p_val_t:.4e}")
print(f"Teste Mann-Whitney: U = {u_stat:.1f}, p-valor = {p_val_u:.4e}")


## 5. Comparação Interdivisões: Série A vs. Série B (2022–2023)

Análise comparativa das temporadas 2022 e 2023 entre as duas principais divisões nacionais (1.520 partidas harmonizadas).


In [ ]:
tabela_07_path = os.path.join(PROJECT_ROOT, "reports", "tables", "tabela_07_comparacao_metricas_series_a_b.csv")
tabela_08_path = os.path.join(PROJECT_ROOT, "reports", "tables", "tabela_08_testes_estatisticos_serie_a_vs_b.csv")

df_tab07 = pd.read_csv(tabela_07_path)
df_tab08 = pd.read_csv(tabela_08_path)

print("Comparação Descritiva Séries A vs. B:")
display(df_tab07)

print("\nTestes Estatísticos de Hipótese (Série A vs. Série B):")
display(df_tab08)


## 6. Gradiente de Dose-Resposta por Nível de Exposição a Apostas

Comparação entre partidas contemporâneas (2019–2024) categorizadas por exposição comercial:
* **Nenhuma:** 0 clubes com patrocínio de aposta;
* **Parcial:** 1 clube com patrocínio;
* **Total:** Ambas as equipes patrocinadas por casas de apostas.


In [ ]:
tabela_05_path = os.path.join(PROJECT_ROOT, "reports", "tables", "tabela_05_comparacao_partidas_por_exposicao.csv")
df_tab05 = pd.read_csv(tabela_05_path)
print("Efeito Dose-Resposta na Partida (2019–2024):")
display(df_tab05)
